# 第5章　利率期限结构与曲线构建

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch05_term_structure.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch05_term_structure.ipynb)

复现例5.1（手算 Bootstrap）、定价自检、线性 vs 样条插值的远期对比、QuantLib 对拍，以及三条曲线案例。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import curve, data, plotting
plotting.use_chinese_style()


## 例5.1　从平价收益率 Bootstrap 即期利率


In [ ]:
par = [0.02, 0.025, 0.028]
zeros, dfs = curve.bootstrap(par)
print(f"{'期限':>4}{'平价c':>9}{'即期z':>10}{'折现DF':>11}")
for n, (c, z, d) in enumerate(zip(par, zeros, dfs), start=1):
    print(f'{n:>4}{c*100:>8.2f}%{z*100:>9.4f}%{d:>11.6f}')
print('\n上行曲线 -> 即期高于平价（z3 > c3）:', round(zeros[-1]*100,4), '>', par[-1]*100)


### 自检：把 bootstrap 出的折现因子代回去给平价债定价，应精确等于面值（编程实验 7）


In [ ]:
for n, c in enumerate(par, start=1):
    pv = c * dfs[:n-1].sum() + (1 + c) * dfs[n-1]
    print(f'{n}yr 平价债重新定价 = {pv:.8f}  (应=1)')


## 线性 vs 三次样条插值：远期曲线的光滑性（编程实验 8）


In [ ]:
# 用一组整数年即期利率，比较两种插值在细网格上的远期
ten = np.arange(1, 11)
par_int = curve.interpolate(data.load_sample('cgb_yield_curve')['tenor'],
                            data.load_sample('cgb_yield_curve')['yield_pct']/100, ten, 'linear')
z_int, _ = curve.bootstrap(par_int)

fine = np.linspace(1, 10, 91)
z_lin = curve.interpolate(ten, z_int, fine, 'linear')
z_cub = curve.interpolate(ten, z_int, fine, 'cubic')

# 由细网格即期求 1 期远期（离散近似）
def fwd_from_zeros(t, z):
    df = (1 + z) ** (-t)
    return (df[:-1] / df[1:]) ** (1 / (t[1:] - t[:-1])) - 1

fig, _ = plotting.new_axes(figsize=(9, 4))
fig.clf()
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(fine, z_lin*100, label='线性'); ax1.plot(fine, z_cub*100, label='三次样条')
ax1.set_title('即期曲线插值'); ax1.set_xlabel('期限'); ax1.set_ylabel('z (%)'); ax1.legend()
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(fine[1:], fwd_from_zeros(fine, z_lin)*100, label='线性->远期')
ax2.plot(fine[1:], fwd_from_zeros(fine, z_cub)*100, label='样条->远期')
ax2.set_title('远期曲线（光滑性对比）'); ax2.set_xlabel('期限'); ax2.legend()
fig.tight_layout()


## 5.7　QuantLib `PiecewiseYieldCurve` 对拍


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.ActualActual(ql.ActualActual.ISDA)
helpers = []
for ty, p in [(1, 0.02), (2, 0.025), (3, 0.028)]:
    sched = ql.Schedule(today, today + ql.Period(ty, ql.Years), ql.Period(ql.Annual),
                        ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted,
                        ql.DateGeneration.Backward, False)
    helpers.append(ql.FixedRateBondHelper(ql.QuoteHandle(ql.SimpleQuote(100.0)), 0, 100.0,
                                          sched, [p], dc, ql.Unadjusted))
yc = ql.PiecewiseLogLinearDiscount(today, helpers, dc)
print(f"{'期限':>4}{'fi 即期':>11}{'QuantLib 即期':>14}")
for t in (1, 2, 3):
    d = today + ql.Period(t, ql.Years)
    zq = yc.zeroRate(d, dc, ql.Compounded, ql.Annual).rate()
    print(f'{t:>4}{zeros[t-1]*100:>10.4f}%{zq*100:>13.4f}%')


## 5.8　案例：中国国债的到期 / 即期 / 远期三条曲线


In [ ]:
cv = data.load_sample('cgb_yield_curve')
ten = np.arange(1, 11)
par_c = curve.interpolate(cv['tenor'], cv['yield_pct']/100, ten, 'linear')
z_c, _ = curve.bootstrap(par_c)
ft, fwd = curve.forward_curve(z_c)

fig, ax = plotting.new_axes()
ax.plot(ten, par_c*100, marker='o', label='到期收益率（平价）')
ax.plot(ten, z_c*100, marker='^', label='即期利率（bootstrap）')
ax.plot(ft, fwd*100, marker='s', ls='--', label='远期利率')
ax.set_xlabel('期限（年）'); ax.set_ylabel('利率 (%)')
ax.set_title('图5-1　到期 / 即期 / 远期三条曲线'); ax.legend()
fig.tight_layout()
print('10yr: par=%.3f%%  spot=%.3f%%  -> 上行曲线下 spot>par' % (par_c[-1]*100, z_c[-1]*100))


---

> 小结：`fi.curve.bootstrap` 自短及长逐期剥离即期利率（定价自检精确回到面值），与 QuantLib `PiecewiseYieldCurve` 一致；
> 上行曲线下三条曲线呈 par < spot < forward。下一部分用这条即期曲线度量利率风险（久期/凸性）。
